# 01 - Summary tables

Reads the raw experiment results in `output/` and writes every table of the report:

| step | writes |
|---|---|
| `summarize.py` part | `work/csv/*.csv`, and `E_summary.csv` / `F_summary.csv` next to the results of the strong-scaling and exchange studies |
| arrange by section | `work/results/s4.*/...` (intermediate copy of the raw files, regenerated here) |
| GPU distributed FFT, collectives, grid sum, atomics, precision, quality, GPU stages, real data on the GPU, GPU MPI library | `work/source/*.csv`, `work/tex/tab_*.tex` |
| weak-scaling stages, CPU gate error terms | `work/source/weak_stages.csv`, `work/source/gate_cpu_terms.csv` |

Run the cells from top to bottom (*Run All*). Needs numpy, pandas and matplotlib; no cluster, no GPU. Afterwards run `02_figures.ipynb` and `03_latex_tables.ipynb`.

In [ ]:
import os, sys
from pathlib import Path
from types import SimpleNamespace

# Work from the repository root (the directory that holds the Makefile), wherever the notebook is opened from.
REPO = Path.cwd().resolve()
while not (REPO / "Makefile").exists() and REPO != REPO.parent:
    REPO = REPO.parent
assert (REPO / "Makefile").exists(), "open this notebook from inside the repository (the directory with the Makefile)"
os.chdir(REPO)

ROOT      = "output"                 # raw experiment results (the results_* directories of implementation/scripts/submit_all.sh)
DATA_REAL = "output/data_real"       # cell-type labels of the real data set (Tabula Sapiens)
TABLES    = "work/csv"             # summary tables written by summarize.py
SRC       = "work/source"          # source tables of the figures and of the LaTeX tables
TEX       = "work/tex"             # the LaTeX tables of the report
FIGS      = "figures"                # the figures of the report
WORK      = "work/results"           # raw files arranged by report section (intermediate, regenerated by notebook 01)
for d in (TABLES, SRC, TEX, FIGS, WORK):
    os.makedirs(d, exist_ok=True)

## 1. Summary tables of the serial / CPU / GPU studies (`summarize.py`)

### Serial, CPU and GPU summary tables

Turns raw result directories (summary.json / timing.csv written by the programs) into the summary tables that the report's figures
and numbers are read from.  No plotting here (see notebook 02); only numpy is needed, scipy only for `realquality`.


`all` expects the result directories produced by implementation/scripts/submit_all.sh (results_vcpu_E400_big/E, results_vcpu_F/F, results_vcpu_weak256, results_place,
results_official2, ...) below `output/`; it writes into `work/csv/`:

    E_summary.csv (copied next to every E directory)   paired            experiment C1, Fig. R2 and the stage tables
    F_summary.csv (next to results_vcpu_F/F)           f_summary         experiment C4, Fig. R5
    fig2_stage_breakdown.csv                           stage_table       experiment C2, Fig. R3 (per-stage ms per iteration, 250k points / 400 boxes)
    comm_model_fit.csv                                 comm_model        experiment C2, Fig. 8 (T(P) = a/P + b fitted on P <= 64)
    weak256_summary.csv, place_summary.csv             weak_place        experiments W1 and C3, Fig. R6 / R4
    official_compare_summary.csv                       official          experiment O1, Fig. R9
    real_scaling.csv                                   real_scaling      experiment R1 (CPU), Fig. 6

In [ ]:
import argparse, collections, csv, glob, json, os, re, statistics as st, sys

In [ ]:
# ----------------------------------------------------------------------------------------------------------------------------------
# small helpers
# ----------------------------------------------------------------------------------------------------------------------------------
def first_existing(*paths):
    """First path that exists (result directories were renamed when they were copied back from the cluster)."""
    for p in paths:
        if os.path.exists(p):
            return p
    return None

In [ ]:
def read_json(path):
    with open(path) as f:
        return json.load(f)

In [ ]:
def write_csv(path, header, rows):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(header)
        w.writerows(rows)

In [ ]:
# ----------------------------------------------------------------------------------------------------------------------------------
# study E: paired, interleaved strong scaling (experiment C1)
# ----------------------------------------------------------------------------------------------------------------------------------
def paired(dirpath, csv_out=None, quiet=False):
    """Per (variant, P): median over rounds of the loop seconds, of the calibration run done right before it (fixed-grid P=1, 30 iterations) and of
    loop/calibration.  Speed-up is taken against fftrep P=1 of the SAME round (numerator and denominator ran within minutes on the same node); median and
    min..max over rounds are reported.  The calibration is a P=1 probe: it removes slow/fast node effects but not contention among many ranks."""
    def loop(d):
        j = read_json(os.path.join(d, "summary.json"))
        return j.get("loop_seconds_excl_kl", j["loop_seconds"]), j

    def load(d):
        try:
            v = re.findall(r"([\d.]+)", open(os.path.join(d, "loadavg.txt")).read().split("\n")[0])
            return float(v[0])
        except Exception:
            return float("nan")

    meas, cal = {}, {}
    for d in sorted(glob.glob(os.path.join(dirpath, "*_r1"))):
        if not os.path.isfile(os.path.join(d, "summary.json")):
            continue
        b = os.path.basename(d)
        m = re.match(r"cal_rd(\d+)_(fftrep|fftdist)_P(\d+)_P1_", b)
        if m:
            cal[(int(m[1]), m[2], int(m[3]))] = loop(d)[0]
            continue
        m = re.match(r"(fftrep|fftdist)_rd(\d+)_P(\d+)_", b)
        if m:
            L, j = loop(d)
            meas[(int(m[2]), m[1], int(m[3]))] = (L, load(d), j.get("plan_total_seconds", 0))
    if not meas:
        sys.exit("no study-E runs found in " + dirpath)
    rounds = sorted({k[0] for k in meas})
    keys = sorted({(k[1], k[2]) for k in meas}, key=lambda x: (x[0] != "fftrep", x[0], x[1]))
    rows = []
    for v, p in keys:
        rs = [r for r in rounds if (r, v, p) in meas]
        L = [meas[(r, v, p)][0] for r in rs]
        C = [cal.get((r, v, p), float("nan")) for r in rs]
        N = [l / c for l, c in zip(L, C)]
        sp = [meas[(r, "fftrep", 1)][0] / meas[(r, v, p)][0] for r in rs if (r, "fftrep", 1) in meas]
        spn = [(meas[(r, "fftrep", 1)][0] / cal[(r, "fftrep", 1)]) / (meas[(r, v, p)][0] / cal[(r, v, p)]) for r in rs
               if (r, "fftrep", 1) in meas and (r, "fftrep", 1) in cal and (r, v, p) in cal]
        f = lambda x: "%.2f (%.2f..%.2f)" % (st.median(x), min(x), max(x)) if x else "-"
        eff = st.median(sp) / p if sp else float("nan")
        rows.append([v, p, len(rs), st.median(L), st.median(C), st.median(N), f(sp), f(spn), eff])
        if not quiet:
            print("%-8s %3d rounds=%d loop=%.1fs cal=%.2fs speedup=%s eff=%.2f" % (v, p, len(rs), st.median(L), st.median(C), f(sp), eff))
    if csv_out:
        write_csv(csv_out, ["variant", "P", "rounds", "loop_s_median", "cal_s_median", "norm_median", "speedup_vs_rep_P1", "speedup_normalised", "efficiency"], rows)
    return rows

In [ ]:
# ----------------------------------------------------------------------------------------------------------------------------------
# study F: coordinate exchange variants (experiment C4)
# ----------------------------------------------------------------------------------------------------------------------------------
def exchange(dirpath, csv_out=None, quiet=False):
    """One row per (P, variant): repeats, loop seconds, steady iteration time, exchange time per iteration (pack+comm+wait+unpack = everything the exchange
    costs on the critical path), its share of the iteration, bytes received per iteration, mean ghost count.  Each variant is compared with F1 (allgather,
    blocking); a difference below the larger relative repeat spread is reported as 'within noise' and must not be read as a win."""
    rx = re.compile(r"^(F\d+(?:poll)?_[a-z]+_[a-z]+)_P(\d+)_(\w+?)_r(\d+)$")
    runs = {}
    for d in sorted(glob.glob(os.path.join(dirpath, "F*"))):
        m = rx.match(os.path.basename(d))
        f = os.path.join(d, "summary.json")
        if not m or not os.path.isfile(f):
            continue
        try:
            j = read_json(f)
        except Exception:
            continue
        runs.setdefault((int(m.group(2)), m.group(1), m.group(3)), []).append(j)

    def spread(v):
        return (max(v) - min(v)) / st.mean(v) if len(v) > 1 else float("nan")

    def ex(j):
        return sum(j.get("steady_exch_" + k, 0) for k in ("pack", "comm", "wait", "unpack"))

    rows = []
    for (P, var, ds), js in sorted(runs.items()):
        loop = [j.get("loop_seconds_excl_kl", j["loop_seconds"]) for j in js]
        it = [j["steady_iter"] for j in js]
        e = [ex(j) for j in js]
        rows.append(dict(P=P, variant=var, data=ds, n=len(js), loop_s=st.mean(loop), loop_spread=spread(loop), iter_ms=1e3 * st.mean(it), iter_spread=spread(it),
                         exch_ms=1e3 * st.mean(e), exch_spread=spread(e), exch_share=st.mean(e) / st.mean(it),
                         recv_MB=st.mean(j["bytes_exch_recv_per_iter"] for j in js) / 1e6, ghost_mean=st.mean(j.get("ghost_mean", 0) for j in js)))
    if not quiet:
        for r in rows:
            print("P=%-3d %-26s n=%d iter=%.1f ms exch=%.2f ms (%.1f%%)" % (r["P"], r["variant"], r["n"], r["iter_ms"], r["exch_ms"], 100 * r["exch_share"]))
        for P in sorted({r["P"] for r in rows}):
            base = next((r for r in rows if r["P"] == P and r["variant"].startswith("F1_")), None)
            if base is None:
                continue
            for r in rows:
                if r["P"] != P or r is base:
                    continue
                d = r["iter_ms"] / base["iter_ms"] - 1
                noise = max(x for x in (r["iter_spread"], base["iter_spread"], 0.0) if x == x)
                print("  P=%-3d %-26s d_iter=%+6.2f%% (noise %4.1f%%) -> %s" % (P, r["variant"], 100 * d, 100 * noise,
                      "within noise" if abs(d) <= noise else ("faster" if d < 0 else "slower")))
    if csv_out and rows:
        os.makedirs(os.path.dirname(os.path.abspath(csv_out)), exist_ok=True)
        with open(csv_out, "w", newline="") as fh:
            w = csv.DictWriter(fh, fieldnames=list(rows[0]))
            w.writeheader()
            w.writerows(rows)
    return rows

In [ ]:
# ----------------------------------------------------------------------------------------------------------------------------------
# stage table of the distributed-FFT strong scaling (experiment C2)
# ----------------------------------------------------------------------------------------------------------------------------------
STAGE_KEYS = ["steady_iter", "steady_fft", "steady_grid_comm", "steady_fft_transpose", "steady_pot_allgather", "steady_exch_comm"]

In [ ]:
def stage_table(cpu, out_csv, study="results_vcpu_E400_big", Ps=(16, 32, 64, 128, 256, 512)):
    """Median over the rounds of the per-stage times (slowest rank in each iteration, steady state) of the fftdist runs of the 250k / 400-box study, in ms per iteration."""
    def val(j, k):
        v = j.get(k)
        if isinstance(v, dict):
            v = v.get("mean", v.get("avg", v.get("median")))
        return v
    rows = []
    for P in Ps:
        js = [read_json(p) for p in glob.glob(f"{cpu}/{study}/E/fftdist_rd*_P{P}_g250k_r1/summary.json")]
        rows.append([P] + [1000 * st.median([val(j, k) for j in js if val(j, k) is not None] or [float("nan")]) for k in STAGE_KEYS])
    write_csv(out_csv, ["P"] + STAGE_KEYS, rows)
    return rows

In [ ]:
# ----------------------------------------------------------------------------------------------------------------------------------
# cost model T(P) = a/P + b (+ c log2 P) fitted on P <= PFIT, extrapolated to larger P (experiment C2)
# ----------------------------------------------------------------------------------------------------------------------------------
MODEL_STUDIES = [("250k/400 boxes", "results_vcpu_E400_big", 100, 2560 ** 2), ("250k/512 boxes", "results_vcpu_E512_big", 100, 3072 ** 2),
                 ("1M/768 boxes", "results_vcpu_E768_1m", 60, 4608 ** 2)]

In [ ]:
def comm_model(cpu, out_csv, pfit=64):
    """Fits T(P) = a/P + b (and a/P + b + c log2 P) to the fftdist loop time per iteration for P <= pfit (relative-error weighting, non-negative
    coefficients) and reports the relative prediction error for P > pfit.  Returns the fits for plotting."""
    import numpy as np
    lines, fits = [], []
    for name, d, iters, M in MODEL_STUDIES:
        try:
            rows = list(csv.DictReader(open(os.path.join(cpu, d, "E_summary.csv"))))
        except Exception as e:
            print("skip", name, e)
            continue
        pts = {int(r["P"]): float(r["loop_s_median"]) / iters for r in rows if r["variant"] == "fftdist"}
        Ps = np.array(sorted(pts))
        T = np.array([pts[p] for p in Ps])
        f = Ps <= pfit
        for model in ("a/P+b", "a/P+b+c*log2P"):
            cols = lambda P: np.column_stack([1 / P, np.ones_like(P, dtype=float)] + ([np.log2(P)] if "log" in model else []))
            X = cols(Ps[f])
            coef, *_ = np.linalg.lstsq(X / T[f, None], np.ones(f.sum()), rcond=None)
            coef = np.maximum(coef, 0)
            rel = (cols(Ps) @ coef - T) / T
            perr = ";".join(f"{p}:{r:+.2f}" for p, r in zip(Ps[~f], rel[~f]))
            c = coef[2] if len(coef) > 2 else 0.0
            lines.append([name, model, "%.4g" % coef[0], "%.4g" % coef[1], "%.4g" % c, "%.3f" % float(np.sqrt(np.mean(rel[f] ** 2))), perr])
            fits.append(dict(name=name, model=model, coef=coef, M=M, P=Ps, T=T, cols=cols))
    write_csv(out_csv, ["study", "model", "a", "b", "c", "rmse_fit_rel", "pred_err_rel_by_P"], lines)
    return fits

In [ ]:
# ----------------------------------------------------------------------------------------------------------------------------------
# weak scaling to 256 ranks (W1) and placement counterfactual (C3)
# ----------------------------------------------------------------------------------------------------------------------------------
def weak_place(cpu, out_weak, out_place):
    g = collections.defaultdict(list)
    for f in glob.glob(os.path.join(cpu, "results_vcpu_weak256", "*", "*", "summary.json")):
        name = os.path.basename(os.path.dirname(f))
        m = re.match(r"(.*)_P(\d+)_g(\d+)_r(\d)", name)
        var, P = m.group(1), int(m.group(2))
        var = {"fixed": "fixed_B1", "sqrtgrid": "sqrtgrid_fftdist"}.get(var, var)       # the P=1 baseline runs carry the short names
        g[(var, P)].append(read_json(f)["seconds_per_iter_steady"])
    rows = []
    for var in ("fixed_B1", "sqrtgrid_fftdist"):
        if (var, 1) not in g:
            continue
        t1 = st.mean(g[(var, 1)])
        for p in sorted(p for (v, p) in g if v == var):
            v = g[(var, p)]
            rows.append([var, p, len(v), "%.4f" % st.mean(v), "%.4f" % min(v), "%.4f" % max(v), "%.3f" % (t1 / st.mean(v))])
    write_csv(out_weak, ["variant", "P", "n_rounds", "sec_per_iter_mean", "min", "max", "weak_efficiency"], rows)
    # placement: P=64 / 128 / 256 on 1..4 nodes; results_place (P=64 only), results_place256 and results_place8 are pooled, medians over rounds as in Fig. R4
    prow = []
    pg = collections.defaultdict(list)
    for d in ("results_place", "results_place256", "results_place8"):
        pf = os.path.join(cpu, d, "place.csv")
        if os.path.isfile(pf):
            for r in csv.DictReader(open(pf)):
                if d == "results_place" and int(r["P"]) != 64:       # P >= 128 comes from results_place256 only (as in the report)
                    continue
                pg[(int(r["P"]), int(r["K"]), int(r["nodes"]))].append(r)
    for (P, K, n), v in sorted(pg.items()):
        f = lambda c: st.median(float(x[c]) for x in v)
        prow.append([P, n, K, len(v)] + ["%.4f" % f(c) for c in ("sec_per_iter_steady", "fft", "fft_transpose", "pot_allgather", "grid_comm", "exch_comm")])
    if prow:
        write_csv(out_place, ["P", "nodes", "ranks_per_node", "n", "sec_per_iter", "fft", "fft_transpose", "pot_allgather", "grid_comm", "exch_comm"], prow)

In [ ]:
# ----------------------------------------------------------------------------------------------------------------------------------
# official FIt-SNE vs ours (O1)
# ----------------------------------------------------------------------------------------------------------------------------------
def official(cpu, out_csv):
    path = first_existing(os.path.join(cpu, "results_official2", "official_compare", "results.csv"))
    if path is None:
        print("no official comparison results"); return
    g = collections.defaultdict(list)
    for r in csv.DictReader(open(path)):
        g[(r["data"], r["impl"], int(r["threads_or_P"]))].append(r)
    mean = lambda v, c: st.mean(float(x[c]) for x in v)
    rows = []
    for d in sorted({k[0] for k in g}):
        base = {i: mean(g[(d, i, 1)], "loop_seconds") for i in ("official", "ours_serial") if (d, i, 1) in g}
        for (dd, impl, p), v in sorted(g.items(), key=lambda t: (t[0][1], t[0][2])):
            if dd != d:
                continue
            b = base.get("official" if impl == "official" else "ours_serial")
            xs = [float(x["loop_seconds"]) for x in v]
            rows.append([d, impl, p, len(v), "%.2f" % mean(v, "loop_seconds"), "%.2f" % min(xs), "%.2f" % max(xs), "%.2f" % (b / mean(v, "loop_seconds")),
                         "%.4f" % mean(v, "trust"), "%.4f" % mean(v, "knn_recall"), "%.4f" % mean(v, "purity"), "%.4f" % mean(v, "kl_common")])
    write_csv(out_csv, ["data", "impl", "threads_or_P", "n", "loop_s_mean", "loop_s_min", "loop_s_max", "speedup_vs_own_1core", "trust", "knn_recall", "purity", "kl_common"], rows)

In [ ]:
# ----------------------------------------------------------------------------------------------------------------------------------
# real data on the CPU (R1): s / iteration of the fixed 540-box runs on ts50k
# ----------------------------------------------------------------------------------------------------------------------------------
def real_scaling(cpu, out_csv):
    """Time per iteration (loop seconds / iterations) of every ts50k run, averaged over the repetitions of one job; job 'a' = single node (300
    iterations), job 'b' = multi node (100 iterations).  T(1) comes from the single-node job."""
    rows = []
    for tag, d in (("a", "results_vcpu_real"), ("b", "results_vcpu_real2")):
        g = collections.defaultdict(list)
        for f in glob.glob(os.path.join(cpu, d, "realcpu", "*", "summary.json")):
            m = re.match(r"(\w+)_P(\d+)_", os.path.basename(os.path.dirname(f)))
            j = read_json(f)
            g[int(m[2])].append(j.get("loop_seconds_excl_kl", j["loop_seconds"]) / j["iters"])
        for p in sorted(g):
            rows.append([tag, p, len(g[p]), "%.4f" % st.mean(g[p])])
    write_csv(out_csv, ["job", "P", "n_reps", "sec_per_iter"], rows)

In [ ]:
# ----------------------------------------------------------------------------------------------------------------------------------
# cell-type level quality of the real-data embeddings (R1, Section "real data")
# ----------------------------------------------------------------------------------------------------------------------------------
def realquality(D, EMB, OUT):
    """Per embedding: (a) over ALL cells the 2-D k-NN label purity (k = 15), per cell type; (b) over 2000 random cells exact feature-space k-NN:
    trustworthiness, kNN recall, label purity (same definitions as `tsne_tools eval`); (c) the most confused cell-type pairs.
    D = directory with {ts,ts200k,ts50k,ts10k}.{feat,labels}.bin and ts.label_names.txt; EMB = directory of embedding files named
    <config>_<data>_<rep>.bin (e.g. P1_ts50k_r1.bin = embedding.bin of run real_P1_ts50k_r1); writes OUT/real_quality.json.  Needs numpy and scipy."""
    import time
    import numpy as np
    from scipy.spatial import cKDTree
    os.makedirs(OUT, exist_ok=True)
    K, M, SEED = 15, 2000, 7

    def feat(t):
        with open(f"{D}/{t}.feat.bin", "rb") as f:
            f.read(8); N, d = np.frombuffer(f.read(16), dtype=np.int64)
            return np.frombuffer(f.read(), dtype=np.float32).reshape(N, d)

    def labels(t):
        with open(f"{D}/{t}.labels.bin", "rb") as f:
            f.read(8); N = np.frombuffer(f.read(8), dtype=np.int64)[0]
            return np.frombuffer(f.read(), dtype=np.int32)[:N]

    def emb(p):
        with open(p, "rb") as f:
            f.read(8); N = np.frombuffer(f.read(8), dtype=np.int64)[0]
            return np.frombuffer(f.read(), dtype=np.float64).reshape(N, 2)

    names = [l.rstrip("\n") for l in open(f"{D}/ts.label_names.txt")]
    configs = {}
    for fn in sorted(os.listdir(EMB)):
        c, t, r = fn[:-4].split("_")
        configs.setdefault(t, []).append((c, r, os.path.join(EMB, fn)))
    res = {}
    for t in ("ts10k", "ts50k", "ts200k", "ts"):
        if t not in configs:
            continue
        X = feat(t); L = labels(t); N = len(L); t0 = time.time()
        rng = np.random.default_rng(SEED)
        q = rng.choice(N, size=min(M, N), replace=False)
        sq = (X.astype(np.float32) ** 2).sum(1)
        embs = {(c, r): emb(p) for c, r, p in configs[t]}
        U_all, U_q = {}, {}
        for key, Y in embs.items():                                  # 2-D neighbours of every cell (purity, confusion) and of the sample (recall, trustworthiness)
            _, idx = cKDTree(Y).query(Y, k=K + 1)
            U_all[key] = idx[:, 1:]
            U_q[key] = U_all[key][q]
        trust_pen = {k: np.zeros(len(q)) for k in embs}
        recall = {k: np.zeros(len(q)) for k in embs}
        for s in range(0, len(q), 50):                               # feature-space neighbours and ranks (exact, chunked)
            qi = q[s:s + 50]
            Dm = sq[None, :] - 2.0 * (X[qi] @ X.T) + sq[qi][:, None]
            Dm[np.arange(len(qi)), qi] = np.inf
            S = np.sort(Dm, axis=1)
            h = np.argpartition(Dm, K, axis=1)[:, :K]
            for a in range(len(qi)):
                hset = set(h[a].tolist())
                for key in embs:
                    u = U_q[key][s + a]
                    recall[key][s + a] = len(hset.intersection(u.tolist())) / K
                    out = [j for j in u.tolist() if j not in hset]
                    if out:
                        r = np.searchsorted(S[a], Dm[a, out]) + 1
                        trust_pen[key][s + a] = float(np.sum(r - K))
        out_t = {}
        for key, Y in embs.items():
            T = 1.0 - 2.0 / (K * (2 * N - 3 * K - 1)) * trust_pen[key].mean()
            pur_all = (L[U_all[key]] == L[:, None]).mean(1)
            per = {int(lab): [int((L == lab).sum()), float(pur_all[L == lab].mean())] for lab in np.unique(L)}
            nb_lab = L[U_all[key]]
            ii, jj = np.nonzero(nb_lab != L[:, None])
            a_l, b_l = L[ii], nb_lab[ii, jj]
            pk = np.minimum(a_l, b_l).astype(np.int64) * 1000 + np.maximum(a_l, b_l)
            u, c = np.unique(pk, return_counts=True)
            top = np.argsort(-c)[:15]
            out_t["_".join(key)] = dict(trustworthiness=float(T), knn_recall=float(recall[key].mean()), purity_sample=float(pur_all[q].mean()), purity_all=float(pur_all.mean()),
                                        per_type=per, confused=[[int(u[i] // 1000), int(u[i] % 1000), int(c[i])] for i in top])
        res[t] = out_t
        print(t, N, f"{time.time() - t0:.0f}s", {k: (round(v['trustworthiness'], 4), round(v['knn_recall'], 3), round(v['purity_all'], 3)) for k, v in out_t.items()}, flush=True)
    json.dump(dict(names=names, results=res), open(f"{OUT}/real_quality.json", "w"))

In [ ]:
# ----------------------------------------------------------------------------------------------------------------------------------
def run_all(cpu, out):
    tabs = out
    os.makedirs(tabs, exist_ok=True)
    for d in sorted(glob.glob(os.path.join(cpu, "results_vcpu_E*"))):
        if os.path.isdir(os.path.join(d, "E")):
            print("== paired", d)
            paired(os.path.join(d, "E"), os.path.join(d, "E_summary.csv"), quiet=True)
    f = os.path.join(cpu, "results_vcpu_F", "F")
    if os.path.isdir(f):
        print("== exchange", f)
        exchange(f, os.path.join(cpu, "results_vcpu_F", "F_summary.csv"), quiet=True)
    f = os.path.join(cpu, "results_vcpu_F_rep", "F")          # C4 repeated 5 times at P=128 (separate job)
    if os.path.isdir(f):
        print("== exchange", f)
        exchange(f, os.path.join(cpu, "results_vcpu_F_rep", "F_summary.csv"), quiet=True)
    stage_table(cpu, os.path.join(tabs, "fig2_stage_breakdown.csv"))
    comm_model(cpu, os.path.join(tabs, "comm_model_fit.csv"))
    weak_place(cpu, os.path.join(tabs, "weak256_summary.csv"), os.path.join(tabs, "place_summary.csv"))
    official(cpu, os.path.join(tabs, "official_compare_summary.csv"))
    real_scaling(cpu, os.path.join(tabs, "real_scaling.csv"))
    print("tables written to", tabs)

In [ ]:
run_all(ROOT, TABLES)

## 2. Arrange the raw files by report section

In [ ]:
# Arrange the raw files that the report uses by report section (the layout read by the generators below).
# Only csv / json / txt / md files are copied.  A directory that does not exist is reported as MISSING and skipped.
import shutil

def first(*names):
    """first existing one of the alternative directory names"""
    for n in names:
        if os.path.exists(f"{ROOT}/{n}"):
            return f"{ROOT}/{n}"
    return f"{ROOT}/{names[0]}"

def cp_(src, dst):
    if not os.path.exists(src):
        print("MISSING", src); return
    for dp, _, fs in os.walk(src):
        for f in fs:
            if f.endswith((".csv", ".json", ".txt", ".md")):
                t = os.path.join(WORK, dst, os.path.relpath(dp, src), f)
                os.makedirs(os.path.dirname(t), exist_ok=True); shutil.copy2(os.path.join(dp, f), t)

def cp_file(src, dst):
    if os.path.isfile(src):
        os.makedirs(os.path.join(WORK, dst), exist_ok=True); shutil.copy2(src, os.path.join(WORK, dst))

shutil.rmtree(WORK, ignore_errors=True)
# 4.1 serial
cp_(first("results_vanda_cpu_v2/serial"), "s4.1_serial_scaling/N_M_scans")
cp_(first("results_vanda_cpu_knn/serial", "results_vanda_cpu_knn"), "s4.1_serial_scaling/K_scan")
# 4.2 correctness
cp_(first("exact_validation_vanda", "results_validation/exact_validation"), "s4.2_correctness/exact_validation_N5000")
cp_(first("results_vcpu_q"), "s4.2_correctness/quality_vs_P_cpu")
# 4.3 CPU strong scaling and communication
for d in ("E400_big", "E512_big", "E768_1m", "E400", "E512", "E200", "E200_big"):
    cp_(first("results_vcpu_" + d), f"s4.3_cpu_strong_scaling/{d}")
cp_(first("results_bgs_poll2"), "s4.3_cpu_strong_scaling/bgs_poll")
cp_file(f"{TABLES}/comm_model_fit.csv", "s4.3_comm_analysis/cost_model")
cp_file(f"{TABLES}/fig2_stage_breakdown.csv", "s4.3_comm_analysis/stage_breakdown_table")
cp_(first("results_place"), "s4.3_comm_analysis/placement");  cp_file(f"{TABLES}/place_summary.csv", "s4.3_comm_analysis/placement")
cp_(first("results_place256"), "s4.3_comm_analysis/placement_256")
cp_(first("results_place8"), "s4.3_comm_analysis/placement_144_216_256")
cp_(first("results_vcpu_F"), "s4.3_comm_analysis/exchange_variants")
cp_(first("results_vcpu_F_rep"), "s4.3_comm_analysis/exchange_variants_P128_5repeats")
cp_(first("results_coll"), "s4.3_comm_analysis/collectives")
# 4.4 CPU weak scaling
cp_(first("results_vcpu_weak256"), "s4.4_cpu_weak_scaling/5k_points_per_rank_to_P256")
cp_(first("results_vcpu_weak512"), "s4.4_cpu_weak_scaling/5k_points_per_rank_P512_attempt")
cp_file(f"{TABLES}/weak256_summary.csv", "s4.4_cpu_weak_scaling/5k_points_per_rank_to_P256")
# 4.5 GPU
for d in ("fx", "fx_2n"):
    cp_(first("results_" + d), f"s4.5_gpu/one_and_two_gpus/results_{d}")
for d in ("fx_device", "fx_device_2n_u", "fx_hpcxhost", "fx_hpcxhost_2n_u"):
    cp_(first("results_" + d), f"s4.5_gpu/transport_host_vs_cuda_aware/results_{d}")
cp_(first("results/C", "results_fx/C"), "s4.5_gpu/write_conflicts")
cp_(first("results_gpu_samegrid"), "s4.5_gpu/same_grid_cpu_gpu_precision")
GD = "s4.5_gpu/distributed_fft"
for src, dst in (("results_gpu_dist", "same_node"), ("results_gpu_dist_grid", "same_node_grid_scan"), ("results_gpu_dist_real", "real_data"),
                 ("results_gpu_dist_2n_v3", "two_nodes"), ("results_gpu_dist_q2n", "quality_two_nodes"),
                 ("results_gpu_dist_big", "capacity_b2048_b2560"), ("results_gpu_dist_cap", "capacity_b2816"), ("results_gpu_dist_cap2", "capacity_b2944")):
    cp_(first(src), f"{GD}/{dst}")
# 4.6 real data
cp_(first("results_vcpu_real"), "s4.6_real_data/cpu_strong_scaling/results_vcpu_real")
cp_(first("results_vcpu_real2"), "s4.6_real_data/cpu_strong_scaling/results_vcpu_real2")
for d in ("fx_real", "fx_realB1"):
    cp_(first("results_" + d), f"s4.6_real_data/gpu/results_{d}")
cp_file(f"{ROOT}/real_quality.json", "s4.6_real_data/quality_by_size_and_type")
# 4.7 official FIt-SNE
cp_(first("results_official2/official_compare"), "s4.7_official_comparison/official_vs_ours")
cp_file(f"{TABLES}/official_compare_summary.csv", "s4.7_official_comparison/official_vs_ours")
# the tables that summarize.py wrote, next to the tables of the other summary generators
for f in os.listdir(TABLES):
    if f.endswith(".csv"):
        cp_file(f"{TABLES}/{f}", "figure_source_tables")
print("arranged by section in", WORK)

## 3. Optional: cell-type quality of the real-data embeddings

In [ ]:
# Optional (REALQ = True): cell-type quality of the real-data embeddings -> output/real_quality.json.
# Needs scipy, the feature / label files data/ts*.feat.bin, data/ts*.labels.bin written by implementation/scripts/prepare.sh,
# and the embedding.bin files of the real-data runs (not part of output/: they are large).  The stored result is output/real_quality.json.
REALQ = False
if REALQ:
    import glob
    EMB = "work/real_emb"; shutil.rmtree(EMB, ignore_errors=True); os.makedirs(EMB)
    def link(res, P, tag):
        for d in sorted(glob.glob(f"{res}/real/real_P{P}_ts*_r*")):
            if os.path.isfile(f"{d}/embedding.bin"):
                os.symlink(os.path.abspath(f"{d}/embedding.bin"), f"{EMB}/{tag}_{os.path.basename(d)[len(f'real_P{P}_'):]}.bin")
    link(f"{ROOT}/results_fx_real", 1, "P1"); link(f"{ROOT}/results_fx_real", 2, "P2s"); link(f"{ROOT}/results_fx_real_2n", 2, "P2x")
    realquality("data", EMB, "work")
    shutil.copy("work/real_quality.json", f"{ROOT}/real_quality.json")

## 4. Tables of the GPU distributed FFT, collectives and grid sum

### Distributed FFT on the GPU, collectives, grid sum

Tables of the report for the distributed FFT on the GPU, the collective operations and the grid-sum variants:
Reads   work/results/s4.5_gpu/distributed_fft/*/dist.csv, quality.csv   (GPU: replicated against distributed FFT, implementation/scripts/gpu_dist_fft.sh)
        work/results/s4.3_comm_analysis/collectives/coll_all.csv       (collectives micro-benchmark, implementation/scripts/coll_bench.sh)
        work/results/s4.3_cpu_strong_scaling/bgs_poll/B/*/summary.json  (grid-sum variants at P=64,128, run_experiments.sh study Bgs)
Writes  work/source/{gpu_dist_fft_summary,gpu_dist_fft_stages,gpu_dist_capacity,coll_summary,gridsum_summary,gpu_dist_quality}.csv
        work/tex/{tab_gpufft,tab_gpucap,tab_coll,tab_gridsum}.tex   
Timing = median over the repeats of the steady-state seconds per iteration (without the first 10 iterations).

In [ ]:
import csv, glob, json, os, re, statistics as st, collections

In [ ]:
R = WORK; GD = f"{R}/s4.5_gpu/distributed_fft"; OUT = SRC; SEC = TEX
os.makedirs(OUT, exist_ok=True)
STAGES = ["fft_stage", "fft_cols", "fft_rows_inv", "fft_transpose", "pot_allgather", "fft_perm_in", "fft_perm_out", "kernel_spec", "grid_comm", "grid_d2h", "grid_h2d",
          "grid_wait", "scatter", "gather", "gpu_wait", "exch_comm"]

In [ ]:
def read_dist(path):
    """dist.csv has two columns called 'fft' (the FFT mode and the FFT stage time): read by position."""
    if not os.path.exists(path): return []
    rd = csv.reader(open(path)); h = next(rd); h[9] = "fft_stage"
    return [dict(zip(h, r)) for r in rd]

In [ ]:
def group(rows):
    g = collections.defaultdict(list)
    for r in rows:
        if r["transport"] == "host": g[(r["data"], int(r["boxes"]), int(r["P"]), r["fft"])].append(r)
    return g

In [ ]:
med = lambda v, c: st.median(float(x[c]) for x in v)
ms = lambda t: 1e3 * t

In [ ]:
def fmt_ms(x):
    return "%.0f" % x if x >= 100 else "%.1f" % x

In [ ]:
one = []                                   # one node: pooled over the three jobs (same_node, grid scan, real data)
for d in ("same_node", "same_node_grid_scan", "real_data"): one += read_dist(f"{GD}/{d}/dist.csv")
two = read_dist(f"{GD}/two_nodes/dist.csv")
G1, G2 = group(one), group(two)

In [ ]:
# ---- summary csv (every configuration, with the stage columns in ms) ----
with open(f"{OUT}/gpu_dist_fft_summary.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["nodes", "data", "boxes", "P", "fft", "reps", "ms_per_iter"] + ["ms_" + s for s in STAGES] + ["gpu_mem_GB"])
    for nodes, G in ((1, G1), (2, G2)):
        for k, v in sorted(G.items()):
            w.writerow([nodes, k[0], k[1], k[2], k[3], len(v), "%.2f" % ms(med(v, "sec_per_iter_steady"))] + ["%.3f" % ms(med(v, s)) for s in STAGES] +
                       ["%.2f" % (float(v[0]["gpu_mem_bytes"]) / 1e9)])

In [ ]:
# ---- table: time per iteration, replicated against distributed FFT ----
cases = [("g250k", 200), ("g250k", 400), ("g250k", 512), ("g1m", 400), ("g1m", 768), ("g1m", 1024), ("g1m", 1280), ("ts200k", 540)]
label = {"g250k": "250k", "g1m": "1M", "ts200k": "200k (real)"}
rows = []
for d, b in cases:
    def t(G, P, fft):
        v = G.get((d, b, P, fft)); return ms(med(v, "sec_per_iter_steady")) if v else None
    r1, r2, r2n, d1, d2, d2n = t(G1, 1, "replicated"), t(G1, 2, "replicated"), t(G2, 2, "replicated"), t(G1, 1, "distributed"), t(G1, 2, "distributed"), t(G2, 2, "distributed")
    f = lambda x: "--" if x is None else fmt_ms(x)
    rows.append([f"{label[d]}, {b}", f(r1), f(r2), f(r2n), f(d1), f(d2), f(d2n), "%.2f" % (r2 / d2), ("%.2f" % (r2n / d2n)) if (r2n and d2n) else "--"])
tex = ["\\begin{table}[H]", "\\centering", "\\small", "\\begin{tabular}{lcccccccc}", "\\toprule",
       " & \\multicolumn{3}{c}{Replicated FFT} & \\multicolumn{3}{c}{Distributed FFT} & \\multicolumn{2}{c}{Speed of dist./repl.} \\\\",
       "\\cmidrule(lr){2-4}\\cmidrule(lr){5-7}\\cmidrule(lr){8-9}",
       "Points, boxes & $P{=}1$ & $P{=}2$ & $P{=}2$, 2 nodes & $P{=}1$ & $P{=}2$ & $P{=}2$, 2 nodes & 1 node & 2 nodes \\\\", "\\midrule"]
tex += [" & ".join(r) + " \\\\" for r in rows]
tex += ["\\bottomrule", "\\end{tabular}",
        "\\caption{Time per iteration (ms) of the GPU program with the replicated and the distributed FFT on identical fixed grids (host-staged transfers, default Open MPI, median of three repeats). $P=2$: two GPUs on one node, or one GPU on each of two nodes. Last two columns: replicated time divided by distributed time at $P=2$ (below 1: distributed is slower). \\texttt{--}: not measured.}",
        "\\label{tab:gpufft}", "\\end{table}", ""]
open(f"{SEC}/tab_gpufft.tex", "w").write("\n".join(tex))

In [ ]:
# ---- stage table for 1M points, 768 boxes, P=2 ----
rows = []; names = [("Whole iteration", "sec_per_iter_steady"), ("FFT: row transforms (all of the FFT if replicated)", "fft_stage"), ("Column and inverse row transforms", None),
                    ("Two transposes (copy to the host, Alltoall, copy back)", "fft_transpose"), ("Potential Allgatherv", "pot_allgather"), ("Kernel spectrum", "kernel_spec"),
                    ("Grid Reduce\\_scatter", "grid_comm"), ("GPU kernels waited for", "gpu_wait")]
# columns: replicated / distributed FFT on two GPUs of one node (pooled runs), the same on two nodes, and the distributed FFT on ONE GPU (pooled runs)
col = {"r1": G1.get(("g1m", 768, 2, "replicated")), "d1": G1.get(("g1m", 768, 2, "distributed")),
       "r2": G2.get(("g1m", 768, 2, "replicated")), "d2": G2.get(("g1m", 768, 2, "distributed")),
       "p1": G1.get(("g1m", 768, 1, "distributed"))}
with open(f"{OUT}/gpu_dist_fft_stages.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["stage", "repl_1node_ms", "dist_1node_ms", "repl_2nodes_ms", "dist_2nodes_ms", "dist_1gpu_ms"])
    for nm, c in names:
        vals = []
        for key in ("r1", "d1", "r2", "d2", "p1"):
            v = col[key]
            vals.append(ms(med(v, "fft_cols") + med(v, "fft_rows_inv")) if c is None else ms(med(v, c)))
        w.writerow([nm] + ["%.1f" % x for x in vals[:4]] + ["%.3f" % vals[4]])   # the one-GPU column keeps 3 decimals (rounded only when printed)

In [ ]:
# ---- capacity ----
cap = read_dist(f"{GD}/capacity_b2048_b2560/dist.csv") + read_dist(f"{GD}/capacity_b2816/dist.csv") + read_dist(f"{GD}/capacity_b2944/dist.csv")
GC = group(cap); rows = []
with open(f"{OUT}/gpu_dist_capacity.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["boxes", "P", "fft", "reps", "ms_per_iter", "gpu_mem_GB"])
    for b in (2048, 2560, 2816, 2944):
        row = [str(b)]
        for P, fft in ((1, "replicated"), (2, "replicated"), (1, "distributed"), (2, "distributed")):
            v = GC.get(("g1m", b, P, fft))
            if v:
                tt, mm = ms(med(v, "sec_per_iter_steady")), float(v[0]["gpu_mem_bytes"]) / 1e9
                row.append("%.0f\\,ms, %.1f\\,GB" % (tt, mm)); w.writerow([b, P, fft, len(v), "%.1f" % tt, "%.2f" % mm])
            else:
                nm = P == 1 and b == 2944
                row.append("\\texttt{--}" if nm else "fails"); w.writerow([b, P, fft, 0, "", ""])
        rows.append(row)
tex = ["\\begin{table}[H]", "\\centering", "\\small", "\\begin{tabular}{lcccc}", "\\toprule",
       " & \\multicolumn{2}{c}{Replicated FFT} & \\multicolumn{2}{c}{Distributed FFT} \\\\", "\\cmidrule(lr){2-3}\\cmidrule(lr){4-5}",
       "Boxes & $P{=}1$ & $P{=}2$ & $P{=}1$ & $P{=}2$ \\\\", "\\midrule"]
tex += [" & ".join(r) + " \\\\" for r in rows]
tex += ["\\bottomrule", "\\end{tabular}",
        "\\caption{Largest grids on the A40 (1M points, one node, host-staged transfers): time per iteration and the largest memory in use on a GPU (the card has $47.7$\\,GB). Two repeats for 2048 and 2560 boxes and one for 2816 and 2944. ``fails'': the run stopped with an allocation error (out of memory or a cuFFT error); \\texttt{--}: not run.}",
        "\\label{tab:gpucap}", "\\end{table}", ""]
open(f"{SEC}/tab_gpucap.tex", "w").write("\n".join(tex))

In [ ]:
# ---- quality of the distributed FFT across two nodes ----
with open(f"{OUT}/gpu_dist_quality.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["where", "data", "boxes", "P", "fft", "seed", "trust", "knn_recall", "purity", "final_kl", "ms_per_iter"])
    for where, path in (("two_nodes", f"{GD}/quality_two_nodes/quality.csv"), ("one_node", f"{GD}/same_node/quality.csv")):
        if os.path.exists(path):
            for r in csv.DictReader(open(path)):
                w.writerow([where, r["data"], r["boxes"], r["P"], r["fft"], r["seed"], r["trust"], r["knn_recall"], r["purity"], r["final_kl"], "%.1f" % ms(float(r["sec_per_iter_total"]))])

In [ ]:
# ---- collectives ----
rows = []; C = list(csv.DictReader(open(f"{R}/s4.3_comm_analysis/collectives/coll_all.csv")))
with open(f"{OUT}/coll_summary.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["P", "ranks_per_node", "collective", "bytes_per_rank", "mean_ms"])
    for r in C: w.writerow([r["P"], r["ranks_per_node"], r["collective"], r["bytes_per_rank"], r["mean_ms"]])
lay = sorted({(int(r["P"]), int(r["ranks_per_node"])) for r in C})
get = lambda P, n, c: [float(r["mean_ms"]) for r in C if int(r["P"]) == P and int(r["ranks_per_node"]) == n and r["collective"] == c][0]
for P, n in lay:
    rows.append([str(P), str(n)] + ["%.1f" % get(P, n, c) for c in ("reduce_scatter", "allgather", "allreduce")])
tex = ["\\begin{table}[H]", "\\centering", "\\small", "\\begin{tabular}{cccccc}", "\\toprule",
       "$P$ & ranks per node & Reduce\\_scatter & Allgather & Allreduce & Reduce\\_scatter + Allgather \\\\", "\\midrule"]
tex += [" & ".join(r + ["%.1f" % (float(r[2]) + float(r[3]))]) + " \\\\" for r in rows]
tex += ["\\bottomrule", "\\end{tabular}",
        "\\caption{Mean time (ms) of one collective on 46\\,MB per rank, the size of the partial grid of the 250k-point problem with 400 boxes, without any computation (20 repeats).}",
        "\\label{tab:coll}", "\\end{table}", ""]
open(f"{SEC}/tab_coll.tex", "w").write("\n".join(tex))

In [ ]:
# ---- grid-sum variants with replicated FFT (Bgs) ----
runs = collections.defaultdict(list)
for f in glob.glob(f"{R}/s4.3_cpu_strong_scaling/bgs_poll/B/*/summary.json"):
    m = re.match(r"(.+)_P(\d+)_g250k_r(\d)$", os.path.basename(os.path.dirname(f)))
    if m: runs[(m.group(1), int(m.group(2)))].append(json.load(open(f)))
names = [("B1_gridblock", "Blocking"), ("B2_gridnonblock", "Non-blocking"), ("B2poll_gridnonblock", "Non-blocking, polling"), ("A4B2_both", "Also non-blocking exchange")]
with open(f"{OUT}/gridsum_summary.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["variant", "P", "reps", "iters", "ms_per_iter", "ms_fft", "ms_grid_comm", "ms_grid_wait", "ms_exch_comm", "ms_exch_wait"])
    for (v, P), L in sorted(runs.items(), key=lambda kv: (kv[0][1], kv[0][0])):
        m = lambda c: 1e3 * st.median((x.get(c) or 0) for x in L)
        w.writerow([v, P, len(L), L[0]["iters"], "%.0f" % m("seconds_per_iter_steady"), "%.0f" % m("steady_fft"), "%.0f" % m("steady_grid_comm"), "%.0f" % m("steady_grid_wait"),
                    "%.0f" % m("steady_exch_comm"), "%.0f" % m("steady_exch_wait")])
rows = []
for v, nm in names:
    row = [nm]
    for P in (64, 128):
        L = runs.get((v, P))
        if not L: row += ["--", "--", "--"]; continue
        m = lambda c: 1e3 * st.median((x.get(c) or 0) for x in L)
        b1 = 1e3 * st.median(x["seconds_per_iter_steady"] for x in runs[("B1_gridblock", P)])
        t = m("seconds_per_iter_steady"); gs = m("steady_grid_comm") + m("steady_grid_wait")
        row += ["%.0f" % t, "%+.1f" % (100 * (t / b1 - 1)) if v != "B1_gridblock" else "--", "%.0f" % gs]
    rows.append(row)
tex = ["\\begin{table}[H]", "\\centering", "\\small", "\\setlength{\\tabcolsep}{4pt}", "\\begin{tabular}{lcccccc}", "\\toprule",
       " & \\multicolumn{3}{c}{$P=64$} & \\multicolumn{3}{c}{$P=128$} \\\\", "\\cmidrule(lr){2-4}\\cmidrule(lr){5-7}",
       "Variant & Iter. (ms) & $\\Delta$ (\\%) & Grid sum (ms) & Iter. (ms) & $\\Delta$ (\\%) & Grid sum (ms) \\\\", "\\midrule"]
tex += [" & ".join(r) + " \\\\" for r in rows]
tex += ["\\bottomrule", "\\end{tabular}",
        "\\caption{Variants of the grid sum with the replicated FFT (250k points, 400 boxes, median of two or three repeats of 20 or 30 iterations). $\\Delta$ is the change of the iteration time against the blocking variant. The grid sum is the time of the Allreduce (blocking) or of the wait that completes it (non-blocking); the FFT takes $6.5$\\,s in all variants. The last row also uses the non-blocking coordinate exchange.}",
        "\\label{tab:gridsum}", "\\end{table}", ""]
open(f"{SEC}/tab_gridsum.tex", "w").write("\n".join(tex))
print("wrote", ", ".join(sorted(os.listdir(OUT))[-6:]), "and work/tex/tab_{gpufft,gpucap,coll,gridsum}.tex")

## 5. Supporting tables (atomics, precision, quality, GPU stages, real data on the GPU)

### Atomics, precision, quality, GPU stages, real data on the GPU

Small tables of Chapter 4 that make its numbers checkable:
Reads   work/results/s4.5_gpu/write_conflicts/{base,near,far}_P1_g1m*_r*/timing.csv         (spread kernel per iteration, GPU, 1M points, adaptive grid)
        work/results/s4.5_gpu/same_grid_cpu_gpu_precision/{samegrid,quality}.csv            (single against double precision, fixed grids)
        work/results/s4.2_correctness/exact_validation_N5000/quality_summary.csv            (exact t-SNE against FIt-SNE and MPI, N=5000)
        work/results/s4.2_correctness/quality_vs_P_cpu/quality/quality.csv                  (quality against P, 100k points)
Writes  work/source/{write_conflicts_summary,precision_summary,quality_summary_N5000,quality_vs_P_summary}.csv
        work/tex/{tab_atomics,tab_precision,tab_quality}.tex

In [ ]:
import csv, glob, os, statistics as st
import pandas as pd

In [ ]:
R = WORK; OUT = SRC; SEC = TEX; os.makedirs(OUT, exist_ok=True)
W = lambda p, tex: open(p, "w").write("\n".join(tex) + "\n")

In [ ]:
# ---------------------------------------------------------------- write conflicts: spread kernel by phase of the run
WC = f"{R}/s4.5_gpu/write_conflicts"; PH = (("0-4", 0, 5), ("5-249", 5, 250), ("300-499", 300, 500))
rows = []
for nm, s, pat in (("near", 2, "near_P1_g1m_near_r*"), ("base", 6, "base_P1_g1m_r*"), ("far", 12, "far_P1_g1m_far_r*")):
    per = {p[0]: [] for p in PH}
    for d in sorted(glob.glob(f"{WC}/{pat}")):
        r = list(csv.reader(open(d + "/timing.csv"))); h = r[0]; r = r[1:]; ic = h.index("scatter")
        for lab, a, b in PH: per[lab].append(1e3 * st.median(float(x[ic]) for x in r[a:b]))
    rows.append([s] + [st.median(per[p[0]]) for p in PH] + [len(per["0-4"])])
with open(f"{OUT}/write_conflicts_summary.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["cluster_spacing_s", "spread_ms_it0_4", "spread_ms_it5_249", "spread_ms_it300_499", "repeats"])
    for r in rows: w.writerow([r[0]] + ["%.2f" % x for x in r[1:4]] + [r[4]])
tex = ["\\begin{table}[H]", "\\centering", "\\small", "\\begin{tabular}{lccc}", "\\toprule",
       "Cluster spacing $s$ & Iterations 0--4 & 5--249 & 300--499 \\\\", "\\midrule"]
tex += ["%d & %.1f & %.1f & %.1f \\\\" % tuple(r[:4]) for r in rows]
tex += ["\\bottomrule", "\\end{tabular}",
        "\\caption{Time of the spread kernel (ms per iteration) on one GPU, 1M points, adaptive grid, in three phases of a 500-iteration run (median over iterations and three repeats). For comparison, a steady iteration with 1M points and a fixed grid takes $101$--$123$\\,ms (Table~\\ref{tab:precision}).}",
        "\\label{tab:atomics}", "\\end{table}", ""]
W(f"{SEC}/tab_atomics.tex", tex)

In [ ]:
# ---------------------------------------------------------------- single against double precision
SG = f"{R}/s4.5_gpu/same_grid_cpu_gpu_precision"
d = pd.read_csv(f"{SG}/samegrid.csv"); q = pd.read_csv(f"{SG}/quality.csv")
T = d.groupby(["data", "boxes", "precision"]).sec_per_iter_steady.median() * 1e3
Q = q.groupby(["data", "boxes", "precision"])[["trust", "final_kl"]].mean()
rows = []
for data, b, nm in (("g250k", 400, "250k, 400"), ("g250k", 512, "250k, 512"), ("g1m", 768, "1M, 768"), ("g1m", 800, "1M, 800")):
    td, tf = T[(data, b, "double")], T[(data, b, "float")]
    if (data, b, "double") in Q.index:
        trust = "%.4f, %.4f" % (Q.loc[(data, b, "double"), "trust"], Q.loc[(data, b, "float"), "trust"])
        kl = "%.3f, %.3f" % (Q.loc[(data, b, "double"), "final_kl"], Q.loc[(data, b, "float"), "final_kl"])
    else: trust = kl = "\\texttt{--}"
    rows.append([nm, td, tf, td / tf, trust, kl])
with open(f"{OUT}/precision_summary.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["problem", "double_ms", "single_ms", "ratio", "trust_double_single", "kl_double_single"])
    for r in rows: w.writerow([r[0], "%.1f" % r[1], "%.1f" % r[2], "%.2f" % r[3], r[4], r[5]])
tex = ["\\begin{table}[H]", "\\centering", "\\small", "\\setlength{\\tabcolsep}{5pt}", "\\begin{tabular}{lcccrr}", "\\toprule",
       "Points, boxes & Double (ms) & Single (ms) & Ratio & Trust. (d, s) & KL (d, s) \\\\", "\\midrule"]
tex += ["%s & %.1f & %.1f & %.2f & %s & %s \\\\" % tuple(r) for r in rows]
tex += ["\\bottomrule", "\\end{tabular}",
        "\\caption{One GPU with double and with single precision on the fixed grids of the CPU runs: time per iteration (median of three repeats) and quality after 1000 iterations (mean of three seeds; Trust.: trustworthiness; d: double, s: single; \\texttt{--}: not measured).}",
        "\\label{tab:precision}", "\\end{table}", ""]
W(f"{SEC}/tab_precision.tex", tex)

In [ ]:
# ---------------------------------------------------------------- quality: exact against FIt-SNE and MPI (N=5000), and against P (100k points)
s = pd.read_csv(f"{R}/s4.2_correctness/exact_validation_N5000/quality_summary.csv")
p = pd.read_csv(f"{R}/s4.2_correctness/quality_vs_P_cpu/quality/quality.csv").groupby("P")[["trustworthiness", "final_kl"]].agg(["mean", "std"])
a = []
for m, P, fft, nm in (("exact", 1, "-", "exact t-SNE"), ("fast", 1, "-", "FIt-SNE, serial"), ("mpi", 4, "distributed", "MPI, $P=4$"), ("mpi", 12, "distributed", "MPI, $P=12$")):
    r = s[(s.method == m) & (s.P == P) & (s.fft == fft)].iloc[0]
    a.append([nm, "%.5f" % r.trust_mean, "%.5f" % r.kl_mean, "%.4f" % r.kl_std])
b = [["$P=%d$" % P, "%.5f" % p.loc[P, ("trustworthiness", "mean")], "%.4f" % p.loc[P, ("final_kl", "mean")], "%.4f" % p.loc[P, ("final_kl", "std")]] for P in (1, 4, 16, 64)]
with open(f"{OUT}/quality_summary_N5000.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["run", "trust_mean", "kl_mean", "kl_std"]); [w.writerow(r) for r in a]
with open(f"{OUT}/quality_vs_P_summary.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["P", "trust_mean", "kl_mean", "kl_std"]); [w.writerow(r) for r in b]
tex = ["\\begin{table}[H]", "\\centering", "\\small", "\\begin{tabular}{lccc}", "\\toprule",
       "Run & Trustworthiness & KL & Std of KL \\\\", "\\midrule", "\\multicolumn{4}{l}{\\emph{$N=5000$, three seeds, 1000 iterations}} \\\\"]
tex += [" & ".join(r) + " \\\\" for r in a]
tex += ["\\midrule", "\\multicolumn{4}{l}{\\emph{100k points, 200 boxes, three seeds, 1000 iterations}} \\\\"]
tex += [" & ".join(r) + " \\\\" for r in b]
tex += ["\\bottomrule", "\\end{tabular}",
        "\\caption{Quality of the embedding (mean over three seeds): exact t-SNE against FIt-SNE and the MPI code (upper block), and the MPI code against $P$ (lower block).}",
        "\\label{tab:quality}", "\\end{table}", ""]
W(f"{SEC}/tab_quality.tex", tex)
# ---------------------------------------------------------------- GPU, distributed FFT: stages and quality (tables made by summarize_new_results.py -> csv)
import io
ST = pd.read_csv(f"{OUT}/gpu_dist_fft_stages.csv")   # written by summarize_new_results.py; column dist_1gpu_ms = distributed FFT on one GPU (same pooled runs as the Table gpufft)
fm = lambda x: "%.0f" % x if x >= 100 else "%.1f" % x if x else "\\texttt{--}"
tex = ["\\begin{table}[H]", "\\centering", "\\small\\setlength{\\tabcolsep}{4pt}", "\\caption{Numbers of Figure~\\ref{fig:gpudistfft}: stages of one iteration on two GPUs (1M points, 768 boxes, blocking grid sum, host-staged transfers, median of three repeats; the first column is the distributed FFT on one GPU, from six runs of two jobs). The stages overlap and do not add up to the iteration time. ``GPU kernels waited for'' is the time the host waits for the GPU.}", "\\label{tab:gpustages}",
       "\\begin{tabular}{p{4.3cm}rrrrr}", "\\toprule",
       " & One GPU & \\multicolumn{2}{c}{One node} & \\multicolumn{2}{c}{Two nodes} \\\\", "\\cmidrule(lr){3-4}\\cmidrule(lr){5-6}",
       "Stage (ms per iteration) & Distr., $P{=}1$ & Replicated & Distributed & Replicated & Distributed \\\\", "\\midrule"]
for i, (_, r) in enumerate(ST.iterrows()):
    v = [r.repl_1node_ms, r.dist_1node_ms, r.repl_2nodes_ms, r.dist_2nodes_ms]
    tex.append(r.stage + " & " + fm(r.dist_1gpu_ms) + " & " + " & ".join(fm(x) for x in v) + " \\\\")
tex += ["\\bottomrule", "\\end{tabular}", "\\end{table}", ""]
W(f"{SEC}/tab_gpustages.tex", tex)
QG = pd.read_csv(f"{OUT}/gpu_dist_quality.csv").groupby(["where", "fft"])[["trust", "final_kl"]].agg(["mean", "min", "max"]); n = pd.read_csv(f"{OUT}/gpu_dist_quality.csv").groupby(["where", "fft"]).size()
tex = ["\\begin{table}[H]", "\\centering", "\\small", "\\begin{tabular}{llccc}", "\\toprule", "GPUs & FFT & Seeds & Trustworthiness & KL \\\\", "\\midrule"]
for (wh, fft), r in QG.iterrows():
    tr = "%.5f" % r[("trust", "mean")] if n[(wh, fft)] == 1 else "%.5f--%.5f" % (r[("trust", "min")], r[("trust", "max")])
    kl = "%.5f" % r[("final_kl", "mean")] if n[(wh, fft)] == 1 else "%.5f--%.5f" % (r[("final_kl", "min")], r[("final_kl", "max")])
    tex.append("%s & %s & %d & %s & %s \\\\" % ("two nodes" if wh == "two_nodes" else "one node", fft, n[(wh, fft)], tr, kl))
tex += ["\\bottomrule", "\\end{tabular}",
        "\\caption{Quality after 1000 iterations with the replicated and the distributed FFT on two GPUs (250k points, 400 boxes; a range where there are two seeds).}",
        "\\label{tab:gpuqual}", "\\end{table}", ""]
W(f"{SEC}/tab_gpuqual.tex", tex)

In [ ]:
# ---------------------------------------------------------------- real data on the GPU: one against two GPUs, blocking against non-blocking grid sum
import json
RG = f"{R}/s4.6_real_data/gpu"
def gm(pat): return 1e3 * st.median(json.load(open(f))["seconds_per_iter_steady"] for f in glob.glob(pat))
rows = []
for ds, nm in (("ts50k", "50k"), ("ts200k", "200k"), ("ts", "483k")):
    p1 = gm(f"{RG}/results_fx_real/real/real_P1_{ds}_r*/summary.json")
    b1 = gm(f"{RG}/results_fx_realB1/realB1/B1_P2_{ds}_r*/summary.json"); b2 = gm(f"{RG}/results_fx_realB1/realB1/B2_P2_{ds}_r*/summary.json")
    rows.append([nm, p1, b1, b2, p1 / b1, p1 / b2])
with open(f"{OUT}/real_gpu_summary.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["cells", "one_gpu_ms", "two_gpus_blocking_ms", "two_gpus_nonblocking_ms", "ratio_blocking", "ratio_nonblocking"])
    for r in rows: w.writerow([r[0]] + ["%.1f" % x for x in r[1:4]] + ["%.2f" % x for x in r[4:]])
tex = ["\\begin{table}[H]", "\\centering", "\\small", "\\begin{tabular}{lccccc}", "\\toprule",
       " & One GPU & \\multicolumn{2}{c}{Two GPUs (ms)} & \\multicolumn{2}{c}{Speed vs one GPU} \\\\", "\\cmidrule(lr){3-4}\\cmidrule(lr){5-6}", "Cells & (ms) & blocking & non-blocking & blocking & non-blocking \\\\", "\\midrule"]
tex += ["%s & %.1f & %.1f & %.1f & %.2f & %.2f \\\\" % tuple(r) for r in rows]
tex += ["\\bottomrule", "\\end{tabular}",
        "\\caption{Real data on the GPU (fixed grid of 540 boxes, median of three repeats): time per iteration, and the speed of two GPUs relative to one (one-GPU time divided by two-GPU time; below 1: slower).}",
        "\\label{tab:realgpu}", "\\end{table}", ""]
W(f"{SEC}/tab_realgpu.tex", tex)
print("wrote write_conflicts_summary, precision_summary, quality_summary_N5000, quality_vs_P_summary and work/tex/tab_{atomics,precision,quality,gpustages,gpuqual,realgpu}.tex")

## 6. GPU MPI-library / transport table

### GPU MPI library and transport (Table gpumpi)

Table `gpumpi`: time per iteration of two GPUs with the replicated FFT for three MPI/transport settings,
with the blocking and the non-blocking grid sum, on one node and on two nodes (1M points, 800 boxes).


Reads   output/<RES>/B/{B1_gridblock,B2_gridnonblock}_P2_g1m_r*/summary.json   (study B of run_experiments.sh, jobs `gpu_P2*`, `gpu_host_*`, `gpu_dev_*`)
          Open MPI, host-staged : results_fx          (one node)   results_fx_2n            (two nodes)
          HPC-X,    host-staged : results_fx_hpcxhost (one node)   results_fx_hpcxhost_2n_u  (two nodes)
          HPC-X,    CUDA-aware  : results_fx_device   (one node)   results_fx_device_2n_u   (two nodes)
Writes  work/source/gpu_mpi_transport.csv and the LaTeX table.  Value = median over the repeats of seconds_per_iter_steady (ms).

In [ ]:
import argparse, csv, glob, json, os, statistics as st

In [ ]:
A = SimpleNamespace(root=ROOT, out=SRC, tex=f"{TEX}/tab_gpumpi.tex")
ROWS = (("Open MPI", "host-staged", "results_fx", "results_fx_2n"),
        ("HPC-X", "host-staged", "results_fx_hpcxhost", "results_fx_hpcxhost_2n_u"),
        ("HPC-X", "\\emph{CUDA-aware}", "results_fx_device", "results_fx_device_2n_u"))

In [ ]:
def med(res, variant):
    v = [json.load(open(f))["seconds_per_iter_steady"] for f in glob.glob(f"{A.root}/{res}/B/{variant}_P2_g1m_r*/summary.json")]
    return 1000 * st.median(v) if v else float("nan")

In [ ]:
out = []
for lib, tr, one, two in ROWS:
    out.append([lib, tr, med(one, "B1_gridblock"), med(one, "B2_gridnonblock"), med(two, "B1_gridblock"), med(two, "B2_gridnonblock")])
os.makedirs(A.out, exist_ok=True)
with open(f"{A.out}/gpu_mpi_transport.csv", "w", newline="") as f:
    w = csv.writer(f, lineterminator="\n")
    w.writerow(["mpi", "transfers", "one_node_blocking_ms", "one_node_nonblocking_ms", "two_nodes_blocking_ms", "two_nodes_nonblocking_ms"])
    for r in out: w.writerow([r[0], r[1].replace("\\emph{", "").replace("}", "")] + ["%.1f" % x for x in r[2:]])
tex = ["\\begin{table}[H]", "\\centering", "\\small",
       "\\caption{Time per iteration (ms) of two GPUs with the \\emph{replicated FFT} (1M points, 800 boxes, median of three repeats), with the \\emph{blocking} and the \\emph{non-blocking} grid sum.}",
       "\\label{tab:gpumpi}", "\\begin{tabular}{llcccc}", "\\toprule",
       " & & \\multicolumn{2}{c}{One node} & \\multicolumn{2}{c}{Two nodes} \\\\", "\\cmidrule(lr){3-4}\\cmidrule(lr){5-6}",
       "MPI library & Transfers & blocking & non-blocking & blocking & non-blocking \\\\", "\\midrule"]
tex += [" & ".join([r[0], r[1]] + ["%.1f" % x for x in r[2:]]) + " \\\\" for r in out]
tex += ["\\bottomrule", "\\end{tabular}", "\\end{table}", ""]
os.makedirs(os.path.dirname(A.tex) or ".", exist_ok=True)
open(A.tex, "w").write("\n".join(tex))
print("wrote", A.tex, "and", f"{A.out}/gpu_mpi_transport.csv")

## 7. Weak-scaling stages

### Stages of the weak-scaling runs (Figure R19, Table weakstages)

Where the time of the growing-grid weak-scaling runs goes (Section 4.5, Figure R19, Table weakstages).


Reads   output/results_vcpu_weak256/Bweak/sqrtgrid_fftdist_P<P>_g*_r<rep>/summary.json     (job `weak5k` of implementation/scripts/submit_all.sh; P = 16, 64, 128, 256)
Writes  work/source/weak_stages.csv   (milliseconds per iteration; each stage time is that of the slowest rank in the iteration, averaged over the steady-state iterations of a run, then averaged over the repeats)
  iter_ms = seconds_per_iter_steady | fft_ms = steady_fft | kernel_spec_ms = steady_kernel_spec | reduce_scatter_ms = steady_grid_comm
  transposes_ms = steady_fft_transpose | allgatherv_ms = steady_pot_allgather | grid_stages_ms = the sum of the last three | grid_share = grid_stages / iter

In [ ]:
import argparse, csv, glob, json, os, re, statistics as st

In [ ]:
A = SimpleNamespace(root=ROOT, out=SRC)

In [ ]:
def val(j, k):
    v = j.get(k)
    return v.get("mean", v.get("avg", v.get("median"))) if isinstance(v, dict) else v

In [ ]:
runs = {}
for f in glob.glob(f"{A.root}/results_vcpu_weak256/Bweak/sqrtgrid_fftdist_P*_g*_r*/summary.json"):
    runs.setdefault(int(re.search(r"_P(\d+)_g", f).group(1)), []).append(json.load(open(f)))
rows = []
for P in sorted(runs):
    m = lambda k: 1000 * st.mean(val(j, k) for j in runs[P])
    it, fft, ks, rs, tr, ag = (m("seconds_per_iter_steady"), m("steady_fft"), m("steady_kernel_spec"), m("steady_grid_comm"),
                               m("steady_fft_transpose"), m("steady_pot_allgather"))
    grid = rs + tr + ag
    rows.append([P, "%.1f" % it, "%.1f" % fft, "%.1f" % ks, "%.1f" % rs, "%.1f" % tr, "%.1f" % ag, "%.1f" % grid, "%.3f" % (grid / it)])
os.makedirs(A.out, exist_ok=True)
with open(f"{A.out}/weak_stages.csv", "w", newline="") as f:
    w = csv.writer(f, lineterminator="\n")
    w.writerow(["P", "iter_ms", "fft_ms", "kernel_spec_ms", "reduce_scatter_ms", "transposes_ms", "allgatherv_ms", "grid_stages_ms", "grid_share"])
    w.writerows(rows)
print("wrote", f"{A.out}/weak_stages.csv", len(rows), "rows")

## 8. CPU gate error terms

### CPU gate error terms

The four error terms of every CPU correctness-gate configuration (Section 4.2, `work/source/gate_cpu_terms.csv`).


Reads   output/results_vcpu_gate/gate/parallel_{replicated,distributed}/<config>/grad.bin      (job `gate_1n`, `gate_2n` of implementation/scripts/submit_all.sh)
        the reference of each set is the single-rank run  <set>/p1/grad.bin   (which the gate itself checks against the serial program)
Runs    `tsne_tools compare-grad ref test` for every configuration (relative L2 error of Z, of the attraction, of the repulsion and of the gradient)
Writes  work/source/gate_cpu_terms.csv   columns: fft,config,P,rel_Z,rel_attr,rel_rep,rel_grad,pass   (pass = all four terms <= 1e-10)

In [ ]:
import argparse, csv, glob, os, re, subprocess

In [ ]:
A = SimpleNamespace(root=ROOT, tools="build/tsne_tools", out=SRC, tol=1e-10)
import shutil
PRE = f"{A.root}/results_vcpu_gate/gate_cpu_terms.csv"   # precomputed result (the gradient files are 120 MB and not part of output/)
rows = []
for fft in ("replicated", "distributed"):
    base = f"{A.root}/results_vcpu_gate/gate/parallel_{fft}"
    ref = f"{base}/p1/grad.bin"
    if not os.path.exists(ref) or not os.path.exists(A.tools):
        print("no reference", ref); continue
    for d in sorted(glob.glob(f"{base}/p*")):
        name = os.path.basename(d)
        if not os.path.isdir(d) or name == "p1" or not os.path.exists(f"{d}/grad.bin"):
            continue
        out = subprocess.run([A.tools, "compare-grad", ref, f"{d}/grad.bin"], capture_output=True, text=True).stdout
        z = float(re.search(r"rel\(Z\)=([0-9.e+-]+)", out).group(1))
        m = re.search(r"attr=([0-9.e+-]+).*rep=([0-9.e+-]+).*grad=([0-9.e+-]+)", out)
        a, r, g = (float(x) for x in m.groups())
        P = int(re.match(r"p(\d+)_", name).group(1))
        rows.append([fft, name, P, "%.4g" % z, "%.4g" % a, "%.4g" % r, "%.4g" % g, all(x <= A.tol for x in (z, a, r, g))])
os.makedirs(A.out, exist_ok=True)
if not rows and os.path.exists(PRE):
    shutil.copy(PRE, f"{A.out}/gate_cpu_terms.csv")
    print("gradient files or build/tsne_tools not available: copied the precomputed", PRE)
else:
    with open(f"{A.out}/gate_cpu_terms.csv", "w", newline="") as f:
        w = csv.writer(f, lineterminator="\n"); w.writerow(["fft", "config", "P", "rel_Z", "rel_attr", "rel_rep", "rel_grad", "pass"]); w.writerows(rows)
    print("wrote", f"{A.out}/gate_cpu_terms.csv", len(rows), "configurations;", sum(1 for r in rows if r[-1]), "pass")